# EDA inicial — vitais sintéticos (Limen Épico 3)

> Protótipo acadêmico — **não** é dispositivo médico.

Este notebook documenta o catálogo de datasets de **referência** (calibração offline)
e inspeciona as fixtures versionadas em `data/fixtures/vitals/`.

**Runtime/CI não baixam** Kaggle/HuggingFace — usam só as fixtures.

## Catálogo

| Papel | Dataset | URL |
| ----- | ------- | --- |
| Primário | Human vital signs | https://www.kaggle.com/datasets/engrarri21/human-vital-signs |
| Eventos (opc.) | Patient Vital Signs and Event Tracking | https://www.kaggle.com/datasets/parmajha/patient-vital-signs-and-event-tracking |
| Deterioração | Hospital Deterioration | https://www.kaggle.com/datasets/tarekmasryo/hospital-deterioration-dataset |

Brutos grandes: `data/raw/` (gitignored). Regenerar fixtures: `python scripts/calibrate_vitals.py`.

In [ ]:
from pathlib import Path
import csv

ROOT = Path("..").resolve()
VITALS = ROOT / "data" / "fixtures" / "vitals"
files = sorted(VITALS.glob("vitals_*.csv"))
print("Fixtures:", [p.name for p in files])

for path in files:
    with path.open(newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    hrs = [float(r["heart_rate"]) for r in rows]
    spo2 = [float(r["spo2"]) for r in rows]
    anomalies = sum(1 for r in rows if r["label"] == "anomaly")
    print(f"{path.name}: n={len(rows)} HR[{min(hrs):.1f},{max(hrs):.1f}] "
          f"SpO2[{min(spo2):.1f},{max(spo2):.1f}] anomalies={anomalies}")

## Notas de calibração

- Faixas basais próximas a vitais de adultos em repouso (literatura / CSVs Kaggle de referência).
- `vitals_medium`: anomalia a partir do minuto ~20 (taquicardia leve + queda de SpO2).
- `vitals_high`: anomalia a partir do minuto ~15 (desvios maiores — alinhado a deterioração).
- Limiares de Risco BAIXO/MEDIO/ALTO: spec `03-caso-vitais-risco.md` (AnomalyEngine no T3.8+).
- Sem PHI: sem CPF, nome, prontuário ou IDs reais.